# Lumen AI — Predictive Channel Policy Evaluation v3

**Purpose:** Evaluate BAU channel assignment versus multiple optimizer outputs using a Databricks-friendly, presentation-ready workflow.

This notebook is designed for the current predictive/optimization table where:
- One row = one customer.
- `cost_ID` identifies the customer.
- BAU channel indicators use the prefix `BAU_`.
- Optimizer outputs use:
  - `full_opt1_...` = all-channel optimizer, unconstrained.
  - `full_opt2_...` = all-channel optimizer, constrained.
  - `out_opt1_...` = outbound-only optimizer, unconstrained.
  - `out_opt2_...` = outbound-only optimizer, constrained.
- Predicted lift columns use `lift_...` by default.
- PRISM and legacy scores are used for decile diagnostics.

The notebook is intentionally written as a sequence of clear analysis cells rather than one black-box function.


In [ ]:
# ===== 1) CONFIG =====

# Source table/dataframe. If a Python variable with this name exists, the notebook uses it.
# Otherwise it tries spark.table(TABLE_NAME).
TABLE_NAME = "lumen_ai_predictive_policy_table"

CUSTOMER_ID_COL = "cost_ID"
CONVERSION_COL = "converted"       # update if your label is "Converted" or another name

# Channel suffixes used in BAU_, lift_, full_opt*, out_opt* columns.
# If your actual table has only 8 channels, remove the unused suffixes here.
OUTBOUND_CHANNELS = ["leads", "email", "direct_mail"]
INBOUND_CHANNELS = ["phone", "web", "mobile", "branch"]
DIGITAL_CHANNELS = ["paid_search", "display", "seo"]

ALL_CHANNELS = OUTBOUND_CHANNELS + INBOUND_CHANNELS + DIGITAL_CHANNELS

# Column prefixes.
BAU_PREFIX = "BAU_"
LIFT_PREFIX = "lift_"

# PRISM + legacy score columns.
# Update the two legacy score names before running.
SCORE_COLUMNS = {
    "PRISM": "prism_any_cc_propensity",
    "Legacy 1": "legacy_model_1_score",
    "Legacy 2": "legacy_model_2_score",
}

# Scenario registry.
# population:
#   "exact_one_bau"      = customers with exactly one BAU channel
#   "bau_outbound_only" = exact-one BAU customers whose BAU channel is Leads/Email/Direct Mail
SCENARIOS = {
    "All-channel unconstrained": {
        "prefix": "full_opt1_",
        "channels": ALL_CHANNELS,
        "population": "exact_one_bau",
        "description": "Theoretical benchmark: optimizer can choose from all observed channels.",
    },
    "All-channel constrained": {
        "prefix": "full_opt2_",
        "channels": ALL_CHANNELS,
        "population": "exact_one_bau",
        "description": "Theoretical constrained benchmark over all observed channels.",
    },
    "Outbound-only unconstrained — full population": {
        "prefix": "out_opt1_",
        "channels": OUTBOUND_CHANNELS,
        "population": "exact_one_bau",
        "description": "Aggressive outbound-only scenario applied to all clean BAU customers.",
    },
    "Outbound-only constrained — full population": {
        "prefix": "out_opt2_",
        "channels": OUTBOUND_CHANNELS,
        "population": "exact_one_bau",
        "description": "Aggressive outbound-only constrained scenario applied to all clean BAU customers.",
    },
    "Outbound-only unconstrained — BAU outbound population": {
        "prefix": "out_opt1_",
        "channels": OUTBOUND_CHANNELS,
        "population": "bau_outbound_only",
        "description": "Most defensible unconstrained outbound reassignment: only customers whose BAU was outbound.",
    },
    "Outbound-only constrained — BAU outbound population": {
        "prefix": "out_opt2_",
        "channels": OUTBOUND_CHANNELS,
        "population": "bau_outbound_only",
        "description": "Most defensible constrained outbound reassignment: only customers whose BAU was outbound.",
    },
}

# Output path for plots. Set to None to skip saving files.
OUTPUT_DIR = "/dbfs/FileStore/lumen_ai_predictive_v3"

# Plot style.
TD_GREEN_DARK = "#005D3B"
TD_GREEN = "#008A00"
TD_GREEN_MED = "#2E7D32"
TD_GREEN_LIGHT = "#A5D6A7"
TD_GREEN_PALE = "#E8F5E9"
TD_GOLD = "#B8860B"
TD_GRAY = "#5F6368"

CHANNEL_COLORS = {
    "leads": "#005D3B",
    "email": "#008A00",
    "direct_mail": "#66A61E",
    "phone": "#2E7D32",
    "web": "#43A047",
    "mobile": "#81C784",
    "branch": "#A5D6A7",
    "paid_search": "#B8860B",
    "display": "#C9A227",
    "seo": "#D6B85A",
}

PLOT_SIZES = {
    "channel_mix": (12, 6),
    "decile_mix": (13, 7),
    "uplift_bars": (8, 5),
    "switch_matrix": (8, 6),
    "rank_table": (12, 5),
}

PLOT_CONFIG = {
    "font_size": 11,
    "title_size": 15,
    "label_size": 11,
    "annotation_size": 10,
    "bar_label_threshold": 0.04,  # only label stacked bar segments above this share
    "dpi": 220,
    "rotate_xticks": 0,
}

# Display names for charts.
CHANNEL_DISPLAY_NAMES = {
    "leads": "Leads",
    "email": "Email",
    "direct_mail": "Direct Mail",
    "phone": "Phone",
    "web": "Web",
    "mobile": "Mobile",
    "branch": "Branch",
    "paid_search": "Paid Search",
    "display": "Display",
    "seo": "SEO",
}


In [ ]:
# ===== 2) Imports, plotting style, and data load =====

from pyspark.sql import functions as F
from pyspark.sql import Window
from functools import reduce
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import math

try:
    import seaborn as sns
except Exception:
    sns = None

if OUTPUT_DIR:
    os.makedirs(OUTPUT_DIR, exist_ok=True)

plt.rcParams.update({
    "font.size": PLOT_CONFIG["font_size"],
    "axes.titlesize": PLOT_CONFIG["title_size"],
    "axes.labelsize": PLOT_CONFIG["label_size"],
    "xtick.labelsize": PLOT_CONFIG["font_size"],
    "ytick.labelsize": PLOT_CONFIG["font_size"],
    "legend.fontsize": PLOT_CONFIG["font_size"],
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#333333",
    "axes.grid": True,
    "grid.alpha": 0.25,
})

def ddisplay(obj):
    """Databricks-friendly display fallback."""
    try:
        display(obj)
    except Exception:
        print(obj)

if TABLE_NAME in globals():
    raw_df = globals()[TABLE_NAME]
else:
    raw_df = spark.table(TABLE_NAME)

print(f"Loaded table/dataframe: {TABLE_NAME}")
print(f"Rows: {raw_df.count():,}")
print(f"Columns: {len(raw_df.columns):,}")


In [ ]:
# ===== 3) Helper functions =====

def safe_col(c):
    return F.coalesce(F.col(c).cast("int"), F.lit(0))

def required_columns_for_prefix(prefix, channels):
    return [f"{prefix}{ch}" for ch in channels]

def validate_required_columns(df, required_cols, context=""):
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns for {context}: {missing}")
    print(f"OK: all required columns present for {context}.")

def n_selected_expr(prefix, channels):
    cols = [safe_col(f"{prefix}{ch}") for ch in channels]
    return reduce(lambda a, b: a + b, cols) if cols else F.lit(0)

def selected_channel_expr(prefix, channels):
    expr = F.lit(None).cast("string")
    # reverse order so first channel in list wins
    for ch in reversed(channels):
        expr = F.when(safe_col(f"{prefix}{ch}") == 1, F.lit(ch)).otherwise(expr)
    return expr

def selected_lift_expr(channel_col):
    expr = F.lit(None).cast("double")
    for ch in reversed(ALL_CHANNELS):
        lift_col = f"{LIFT_PREFIX}{ch}"
        if lift_col in working_df.columns:
            expr = F.when(F.col(channel_col) == ch, F.col(lift_col).cast("double")).otherwise(expr)
    return expr

def add_score_decile_non_null(df, score_col, out_col):
    """Decile 1 = highest score. Null scores remain null."""
    if score_col not in df.columns:
        print(f"WARNING: score column not found: {score_col}. Skipping {out_col}.")
        return df.withColumn(out_col, F.lit(None).cast("int"))
    non_null = (
        df.filter(F.col(score_col).isNotNull())
          .withColumn(out_col, F.ntile(10).over(Window.orderBy(F.col(score_col).desc())))
    )
    nulls = df.filter(F.col(score_col).isNull()).withColumn(out_col, F.lit(None).cast("int"))
    return non_null.unionByName(nulls)

def clean_channel_name(ch):
    return CHANNEL_DISPLAY_NAMES.get(ch, ch.replace("_", " ").title())

def format_pct(x, decimals=1):
    if x is None or pd.isna(x):
        return ""
    return f"{100*x:.{decimals}f}%"

def save_plot(name):
    if OUTPUT_DIR:
        path = f"{OUTPUT_DIR}/{name}.png"
        plt.savefig(path, dpi=PLOT_CONFIG["dpi"], bbox_inches="tight")
        print(f"Saved: {path}")

def policy_mix_pdf(df_spark, policy_col, channels=None):
    if channels is None:
        channels = ALL_CHANNELS
    pdf = (
        df_spark
        .groupBy(policy_col)
        .agg(F.count("*").alias("customers"))
        .toPandas()
    )
    total = pdf["customers"].sum()
    pdf["share"] = pdf["customers"] / total if total else 0
    pdf = pdf.rename(columns={policy_col: "channel"})
    pdf["channel"] = pdf["channel"].fillna("missing")
    order = [c for c in channels if c in pdf["channel"].tolist()] + [c for c in pdf["channel"].tolist() if c not in channels]
    pdf["channel"] = pd.Categorical(pdf["channel"], categories=order, ordered=True)
    return pdf.sort_values("channel").reset_index(drop=True)

def plot_policy_mix_bar(mix_pdf, title, figsize=None):
    if figsize is None:
        figsize = PLOT_SIZES["channel_mix"]
    plot_pdf = mix_pdf.copy()
    plot_pdf["label"] = plot_pdf["channel"].astype(str).map(clean_channel_name)
    colors = [CHANNEL_COLORS.get(str(ch), TD_GRAY) for ch in plot_pdf["channel"].astype(str)]
    fig, ax = plt.subplots(figsize=figsize)
    bars = ax.bar(plot_pdf["label"], plot_pdf["share"], color=colors)
    ax.set_title(title, pad=14, fontweight="bold", color=TD_GREEN_DARK)
    ax.set_ylabel("Customer share")
    ax.set_ylim(0, max(plot_pdf["share"].max() * 1.25, 0.05))
    ax.yaxis.set_major_formatter(lambda x, pos: f"{100*x:.0f}%")
    for bar, share, n in zip(bars, plot_pdf["share"], plot_pdf["customers"]):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(), f"{100*share:.1f}%\n({n:,.0f})",
                ha="center", va="bottom", fontsize=PLOT_CONFIG["annotation_size"])
    ax.tick_params(axis="x", rotation=PLOT_CONFIG["rotate_xticks"])
    plt.tight_layout()
    return fig, ax

def decile_policy_share_pdf(df_spark, decile_col, policy_col, channels=None, include_missing_score=False):
    if channels is None:
        channels = ALL_CHANNELS
    temp = df_spark
    if not include_missing_score:
        temp = temp.filter(F.col(decile_col).isNotNull())
    agg = (
        temp
        .withColumn("decile_label", F.when(F.col(decile_col).isNull(), F.lit("Missing")).otherwise(F.col(decile_col).cast("string")))
        .groupBy("decile_label", policy_col)
        .agg(F.count("*").alias("customers"))
    )
    pdf = agg.toPandas().rename(columns={policy_col: "channel"})
    if pdf.empty:
        return pdf
    totals = pdf.groupby("decile_label")["customers"].transform("sum")
    pdf["share"] = pdf["customers"] / totals
    # ordering
    def dec_order(x):
        return 999 if x == "Missing" else int(x)
    pdf["decile_order"] = pdf["decile_label"].map(dec_order)
    pdf["channel"] = pdf["channel"].fillna("missing")
    pdf = pdf.sort_values(["decile_order", "channel"]).reset_index(drop=True)
    return pdf

def plot_stacked_decile_share(decile_pdf, title, channels=None, figsize=None):
    if figsize is None:
        figsize = PLOT_SIZES["decile_mix"]
    if channels is None:
        channels = ALL_CHANNELS
    if decile_pdf.empty:
        print("No data to plot.")
        return None, None
    pivot = decile_pdf.pivot_table(index="decile_label", columns="channel", values="share", fill_value=0, aggfunc="sum")
    # keep desired order
    ordered_cols = [c for c in channels if c in pivot.columns] + [c for c in pivot.columns if c not in channels]
    pivot = pivot[ordered_cols]
    # sort deciles
    dec_order = sorted(pivot.index, key=lambda x: 999 if x == "Missing" else int(x))
    pivot = pivot.loc[dec_order]
    fig, ax = plt.subplots(figsize=figsize)
    bottom = np.zeros(len(pivot))
    x = np.arange(len(pivot))
    for ch in pivot.columns:
        vals = pivot[ch].values
        color = CHANNEL_COLORS.get(str(ch), TD_GRAY)
        ax.bar(x, vals, bottom=bottom, label=clean_channel_name(str(ch)), color=color)
        for i, v in enumerate(vals):
            if v >= PLOT_CONFIG["bar_label_threshold"]:
                ax.text(i, bottom[i] + v/2, f"{100*v:.0f}%", ha="center", va="center", fontsize=9, color="white")
        bottom += vals
    ax.set_title(title, pad=14, fontweight="bold", color=TD_GREEN_DARK)
    ax.set_xlabel("Score decile (1 = highest score)")
    ax.set_ylabel("Customer share")
    ax.set_xticks(x)
    ax.set_xticklabels(pivot.index)
    ax.yaxis.set_major_formatter(lambda y, pos: f"{100*y:.0f}%")
    ax.legend(title="Channel", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    return fig, ax


In [ ]:
# ===== 4) Validate base columns and compute BAU treatment counts =====

base_required = [CUSTOMER_ID_COL, CONVERSION_COL]
validate_required_columns(raw_df, base_required, "base columns")
validate_required_columns(raw_df, required_columns_for_prefix(BAU_PREFIX, ALL_CHANNELS), "BAU channels")
validate_required_columns(raw_df, [f"{LIFT_PREFIX}{ch}" for ch in ALL_CHANNELS], "lift columns")

for scenario_name, sc in SCENARIOS.items():
    validate_required_columns(raw_df, required_columns_for_prefix(sc["prefix"], sc["channels"]), scenario_name)

working_df = raw_df.withColumn(CONVERSION_COL, F.col(CONVERSION_COL).cast("double"))

working_df = working_df.withColumn("n_bau_channels", n_selected_expr(BAU_PREFIX, ALL_CHANNELS))

bau_count_sdf = (
    working_df
    .groupBy("n_bau_channels")
    .agg(
        F.count("*").alias("customers"),
        F.avg(CONVERSION_COL).alias("conversion_rate")
    )
    .withColumn("customer_share", F.col("customers") / F.sum("customers").over(Window.partitionBy()))
    .orderBy("n_bau_channels")
)

ddisplay(bau_count_sdf)

print("Interpretation:")
print("- n_bau_channels = 1 is the clean apples-to-apples subset because the optimizer assigns exactly one channel.")
print("- 0 or 2+ BAU channel customers are excluded from the core predictive-policy comparison.")


In [ ]:
# ===== 5) Keep exact-one BAU customers and add selected policy columns =====

exact_one_df = (
    working_df
    .filter(F.col("n_bau_channels") == 1)
    .withColumn("BAU_selected_channel", selected_channel_expr(BAU_PREFIX, ALL_CHANNELS))
)

# Add scenario selected-channel columns.
for scenario_name, sc in SCENARIOS.items():
    col_safe = scenario_name.lower().replace(" ", "_").replace("—", "_").replace("-", "_").replace("/", "_")
    selected_col = f"{col_safe}__selected_channel"
    exact_one_df = exact_one_df.withColumn(selected_col, selected_channel_expr(sc["prefix"], sc["channels"]))

# Add selected lift for BAU and scenarios.
exact_one_df = exact_one_df.withColumn("BAU_selected_lift", selected_lift_expr("BAU_selected_channel"))

for scenario_name, sc in SCENARIOS.items():
    col_safe = scenario_name.lower().replace(" ", "_").replace("—", "_").replace("-", "_").replace("/", "_")
    selected_col = f"{col_safe}__selected_channel"
    lift_col = f"{col_safe}__selected_lift"
    exact_one_df = exact_one_df.withColumn(lift_col, selected_lift_expr(selected_col))

# Population flags.
exact_one_df = exact_one_df.withColumn(
    "is_bau_outbound",
    F.col("BAU_selected_channel").isin(OUTBOUND_CHANNELS)
)

# Score deciles.
for score_name, score_col in SCORE_COLUMNS.items():
    safe_name = score_name.lower().replace(" ", "_")
    exact_one_df = add_score_decile_non_null(exact_one_df, score_col, f"{safe_name}_decile")

print(f"Exact-one BAU customers: {exact_one_df.count():,}")
print(f"BAU outbound-only customers: {exact_one_df.filter(F.col('is_bau_outbound')).count():,}")
ddisplay(exact_one_df.select(CUSTOMER_ID_COL, CONVERSION_COL, "BAU_selected_channel", "BAU_selected_lift", "is_bau_outbound").limit(10))


In [ ]:
# ===== 6) Score coverage table: PRISM vs legacy model availability =====

coverage_rows = []
total_n = exact_one_df.count()

for score_name, score_col in SCORE_COLUMNS.items():
    if score_col in exact_one_df.columns:
        non_null_n = exact_one_df.filter(F.col(score_col).isNotNull()).count()
        missing_n = total_n - non_null_n
        coverage_rows.append({
            "score": score_name,
            "column": score_col,
            "customers": total_n,
            "non_null_score": non_null_n,
            "missing_score": missing_n,
            "coverage": non_null_n / total_n if total_n else None,
        })
    else:
        coverage_rows.append({
            "score": score_name,
            "column": score_col,
            "customers": total_n,
            "non_null_score": 0,
            "missing_score": total_n,
            "coverage": 0.0,
        })

coverage_pdf = pd.DataFrame(coverage_rows)
ddisplay(spark.createDataFrame(coverage_pdf))


In [ ]:
# ===== 7) BAU channel distribution by PRISM / legacy deciles =====
# This answers: "Does BAU look more aligned with legacy scores than with PRISM?"

for score_name in SCORE_COLUMNS.keys():
    safe_name = score_name.lower().replace(" ", "_")
    decile_col = f"{safe_name}_decile"
    pdf = decile_policy_share_pdf(
        exact_one_df,
        decile_col=decile_col,
        policy_col="BAU_selected_channel",
        channels=ALL_CHANNELS,
        include_missing_score=True,
    )
    ddisplay(spark.createDataFrame(pdf) if not pdf.empty else pdf)
    fig, ax = plot_stacked_decile_share(
        pdf,
        title=f"BAU channel mix by {score_name} decile",
        channels=ALL_CHANNELS,
        figsize=PLOT_SIZES["decile_mix"],
    )
    if fig:
        save_plot(f"bau_channel_mix_by_{safe_name}_decile")
        plt.show()

print("Message to take from this section:")
print("- If BAU Leads looks scattered across PRISM deciles but concentrated in high legacy deciles, BAU may have been driven by legacy model scores at the time.")


In [ ]:
# ===== 8) Optimizer allocation by score decile for each scenario =====

for scenario_name, sc in SCENARIOS.items():
    col_safe = scenario_name.lower().replace(" ", "_").replace("—", "_").replace("-", "_").replace("/", "_")
    selected_col = f"{col_safe}__selected_channel"
    pop_df = exact_one_df if sc["population"] == "exact_one_bau" else exact_one_df.filter(F.col("is_bau_outbound"))
    for score_name in SCORE_COLUMNS.keys():
        safe_score = score_name.lower().replace(" ", "_")
        decile_col = f"{safe_score}_decile"
        pdf = decile_policy_share_pdf(
            pop_df,
            decile_col=decile_col,
            policy_col=selected_col,
            channels=sc["channels"],
            include_missing_score=True,
        )
        fig, ax = plot_stacked_decile_share(
            pdf,
            title=f"{scenario_name}\nchannel mix by {score_name} decile",
            channels=sc["channels"],
            figsize=PLOT_SIZES["decile_mix"],
        )
        if fig:
            save_plot(f"{col_safe}_mix_by_{safe_score}_decile")
            plt.show()


In [ ]:
# ===== 9) BAU observed conversion rate by score decile × channel =====
# This is a sanity check. It does NOT prove causality, but it shows historical performance patterns.

for score_name in SCORE_COLUMNS.keys():
    safe_score = score_name.lower().replace(" ", "_")
    decile_col = f"{safe_score}_decile"
    conv_sdf = (
        exact_one_df
        .withColumn("decile_label", F.when(F.col(decile_col).isNull(), F.lit("Missing")).otherwise(F.col(decile_col).cast("string")))
        .groupBy("decile_label", "BAU_selected_channel")
        .agg(
            F.count("*").alias("customers"),
            F.sum(CONVERSION_COL).alias("conversions"),
            F.avg(CONVERSION_COL).alias("conversion_rate"),
            F.avg("BAU_selected_lift").alias("avg_bau_lift"),
        )
        .orderBy("decile_label", "BAU_selected_channel")
    )
    print(f"\nBAU conversion by channel and {score_name} decile")
    ddisplay(conv_sdf)


In [ ]:
# ===== 10) Average model-predicted lift by score decile × channel =====
# This checks whether the optimizer's allocation is consistent with the model score surface.

def avg_lift_by_decile_channel(input_df, decile_col, channels):
    exprs = [F.avg(F.col(f"{LIFT_PREFIX}{ch}").cast("double")).alias(ch) for ch in channels if f"{LIFT_PREFIX}{ch}" in input_df.columns]
    return (
        input_df
        .withColumn("decile_label", F.when(F.col(decile_col).isNull(), F.lit("Missing")).otherwise(F.col(decile_col).cast("string")))
        .groupBy("decile_label")
        .agg(*exprs)
        .orderBy("decile_label")
    )

for score_name in SCORE_COLUMNS.keys():
    safe_score = score_name.lower().replace(" ", "_")
    decile_col = f"{safe_score}_decile"
    print(f"\nAverage predicted lift by channel and {score_name} decile")
    lift_sdf = avg_lift_by_decile_channel(exact_one_df, decile_col, ALL_CHANNELS)
    ddisplay(lift_sdf)


In [ ]:
# ===== 11) Scenario-level match/non-match and expected conversion impact =====

summary_rows = []

for scenario_name, sc in SCENARIOS.items():
    col_safe = scenario_name.lower().replace(" ", "_").replace("—", "_").replace("-", "_").replace("/", "_")
    selected_col = f"{col_safe}__selected_channel"
    opt_lift_col = f"{col_safe}__selected_lift"
    pop_df = exact_one_df if sc["population"] == "exact_one_bau" else exact_one_df.filter(F.col("is_bau_outbound"))

    temp = (
        pop_df
        .withColumn("is_match", F.col("BAU_selected_channel") == F.col(selected_col))
        .withColumn("incremental_lift_vs_bau", F.col(opt_lift_col) - F.col("BAU_selected_lift"))
    )

    row_all = temp.agg(
        F.count("*").alias("customers"),
        F.sum(CONVERSION_COL).alias("bau_conversions"),
        F.avg(CONVERSION_COL).alias("bau_conversion_rate"),
        F.avg("BAU_selected_lift").alias("avg_bau_lift"),
        F.avg(opt_lift_col).alias("avg_optimizer_lift"),
        F.avg("incremental_lift_vs_bau").alias("avg_incremental_lift"),
        F.sum("incremental_lift_vs_bau").alias("expected_incremental_conversions"),
        F.avg(F.col("is_match").cast("double")).alias("match_rate"),
    ).first().asDict()

    row_nonmatch = temp.filter(~F.col("is_match")).agg(
        F.count("*").alias("nonmatch_customers"),
        F.sum(CONVERSION_COL).alias("nonmatch_bau_conversions"),
        F.avg(CONVERSION_COL).alias("nonmatch_bau_conversion_rate"),
        F.avg("BAU_selected_lift").alias("nonmatch_avg_bau_lift"),
        F.avg(opt_lift_col).alias("nonmatch_avg_optimizer_lift"),
        F.avg("incremental_lift_vs_bau").alias("nonmatch_avg_incremental_lift"),
        F.sum("incremental_lift_vs_bau").alias("nonmatch_expected_incremental_conversions"),
    ).first().asDict()

    customers = row_all["customers"] or 0
    bau_cr = row_all["bau_conversion_rate"] or 0
    avg_delta = row_all["avg_incremental_lift"] or 0
    expected_cr = bau_cr + avg_delta
    rel_uplift = avg_delta / bau_cr if bau_cr else None

    nonmatch_bau_cr = row_nonmatch["nonmatch_bau_conversion_rate"] or 0
    nonmatch_delta = row_nonmatch["nonmatch_avg_incremental_lift"] or 0
    nonmatch_expected_cr = nonmatch_bau_cr + nonmatch_delta
    nonmatch_rel = nonmatch_delta / nonmatch_bau_cr if nonmatch_bau_cr else None

    summary_rows.append({
        "scenario": scenario_name,
        "population": sc["population"],
        "customers": customers,
        "bau_conversions": row_all["bau_conversions"],
        "bau_conversion_rate": bau_cr,
        "match_rate": row_all["match_rate"],
        "avg_bau_lift": row_all["avg_bau_lift"],
        "avg_optimizer_lift": row_all["avg_optimizer_lift"],
        "avg_incremental_lift_pp": avg_delta,
        "expected_optimizer_conversion_rate": expected_cr,
        "relative_expected_uplift": rel_uplift,
        "expected_incremental_conversions": row_all["expected_incremental_conversions"],
        "nonmatch_customers": row_nonmatch["nonmatch_customers"],
        "nonmatch_bau_conversion_rate": nonmatch_bau_cr,
        "nonmatch_avg_incremental_lift_pp": nonmatch_delta,
        "nonmatch_expected_conversion_rate": nonmatch_expected_cr,
        "nonmatch_relative_expected_uplift": nonmatch_rel,
        "nonmatch_expected_incremental_conversions": row_nonmatch["nonmatch_expected_incremental_conversions"],
        "description": sc["description"],
    })

scenario_summary_pdf = pd.DataFrame(summary_rows)
ddisplay(spark.createDataFrame(scenario_summary_pdf))

print("Important wording:")
print("- Expected conversion impact is model-estimated from historical data.")
print("- It is not realized uplift until tested live through champion/challenger or A/B validation.")


In [ ]:
# ===== 12) Presentable expected conversion impact plot =====

impact_pdf = scenario_summary_pdf.copy()
impact_pdf["scenario_short"] = impact_pdf["scenario"].str.replace(" — ", "\n", regex=False)
impact_pdf = impact_pdf.sort_values("expected_optimizer_conversion_rate", ascending=False)

fig, ax = plt.subplots(figsize=PLOT_SIZES["uplift_bars"])
colors = [TD_GREEN_DARK if "BAU outbound" in s else TD_GREEN_MED if "Outbound-only constrained" in s else TD_GREEN_LIGHT for s in impact_pdf["scenario"]]
bars = ax.barh(impact_pdf["scenario_short"], impact_pdf["expected_optimizer_conversion_rate"], color=colors)
ax.axvline(impact_pdf["bau_conversion_rate"].iloc[0], color=TD_GRAY, linestyle="--", label="Observed BAU baseline")
ax.set_title("Model-estimated expected conversion impact by optimizer scenario", fontweight="bold", color=TD_GREEN_DARK, pad=14)
ax.set_xlabel("Expected conversion rate")
ax.xaxis.set_major_formatter(lambda x, pos: f"{100*x:.1f}%")
for bar, val, inc in zip(bars, impact_pdf["expected_optimizer_conversion_rate"], impact_pdf["expected_incremental_conversions"]):
    ax.text(bar.get_width(), bar.get_y()+bar.get_height()/2, f" {100*val:.2f}%\n +{inc:,.0f} conv.", va="center", fontsize=9)
ax.legend(loc="lower right")
plt.tight_layout()
save_plot("expected_conversion_impact_by_scenario")
plt.show()


In [ ]:
# ===== 13) Switch matrices: BAU channel → optimizer channel =====

def switch_matrix_pdf(input_df, scenario_name):
    sc = SCENARIOS[scenario_name]
    col_safe = scenario_name.lower().replace(" ", "_").replace("—", "_").replace("-", "_").replace("/", "_")
    selected_col = f"{col_safe}__selected_channel"
    pop_df = input_df if sc["population"] == "exact_one_bau" else input_df.filter(F.col("is_bau_outbound"))
    sdf = (
        pop_df
        .groupBy("BAU_selected_channel", selected_col)
        .agg(F.count("*").alias("customers"))
        .toPandas()
        .rename(columns={selected_col: "optimizer_channel"})
    )
    mat = sdf.pivot_table(index="BAU_selected_channel", columns="optimizer_channel", values="customers", fill_value=0, aggfunc="sum")
    # row-normalized share
    mat_share = mat.div(mat.sum(axis=1).replace(0, np.nan), axis=0)
    return mat, mat_share

for scenario_name in SCENARIOS.keys():
    mat, mat_share = switch_matrix_pdf(exact_one_df, scenario_name)
    print(f"\nSwitch matrix counts: {scenario_name}")
    ddisplay(spark.createDataFrame(mat.reset_index()) if not mat.empty else mat)
    print(f"\nSwitch matrix row shares: {scenario_name}")
    ddisplay(spark.createDataFrame(mat_share.reset_index()) if not mat_share.empty else mat_share)

    if mat_share.empty:
        continue

    fig, ax = plt.subplots(figsize=PLOT_SIZES["switch_matrix"])
    im = ax.imshow(mat_share.values, aspect="auto", cmap="Greens", vmin=0, vmax=np.nanmax(mat_share.values))
    ax.set_xticks(np.arange(len(mat_share.columns)))
    ax.set_yticks(np.arange(len(mat_share.index)))
    ax.set_xticklabels([clean_channel_name(c) for c in mat_share.columns], rotation=45, ha="right")
    ax.set_yticklabels([clean_channel_name(c) for c in mat_share.index])
    ax.set_title(f"BAU → Optimizer channel switch matrix\n{scenario_name}", fontweight="bold", color=TD_GREEN_DARK, pad=14)
    ax.set_xlabel("Optimizer-selected channel")
    ax.set_ylabel("BAU channel")
    for i in range(mat_share.shape[0]):
        for j in range(mat_share.shape[1]):
            val = mat_share.values[i, j]
            if not np.isnan(val) and val >= 0.02:
                ax.text(j, i, f"{100*val:.0f}%", ha="center", va="center", fontsize=8,
                        color="white" if val > 0.35 else "black")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Row share")
    plt.tight_layout()
    save_plot(f"switch_matrix_{scenario_name.lower().replace(' ', '_').replace('—','_')}")
    plt.show()


In [ ]:
# ===== 14) Optional: non-outbound BAU population diagnostic =====
# This does NOT claim we can replace non-outbound interactions.
# It simply sizes the population and shows what the outbound optimizer would choose if treated as an overlay opportunity.

non_outbound_df = exact_one_df.filter(~F.col("is_bau_outbound"))

non_outbound_summary = non_outbound_df.groupBy("BAU_selected_channel").agg(
    F.count("*").alias("customers"),
    F.avg(CONVERSION_COL).alias("observed_conversion_rate"),
    F.avg("BAU_selected_lift").alias("avg_bau_lift"),
).orderBy(F.desc("customers"))

print("Non-outbound BAU population summary")
ddisplay(non_outbound_summary)

for scenario_name in ["Outbound-only unconstrained — full population", "Outbound-only constrained — full population"]:
    sc = SCENARIOS[scenario_name]
    col_safe = scenario_name.lower().replace(" ", "_").replace("—", "_").replace("-", "_").replace("/", "_")
    selected_col = f"{col_safe}__selected_channel"
    mix_pdf = policy_mix_pdf(non_outbound_df, selected_col, channels=OUTBOUND_CHANNELS)
    fig, ax = plot_policy_mix_bar(
        mix_pdf,
        title=f"Exploratory outbound overlay mix for non-outbound BAU customers\n{scenario_name}",
        figsize=PLOT_SIZES["channel_mix"],
    )
    save_plot(f"exploratory_non_outbound_overlay_{col_safe}")
    plt.show()

print("Caveat:")
print("- This is an exploratory overlay view.")
print("- It should not be interpreted as replacing web/search/display/branch/mobile behavior unless the model explicitly scores BAU + outbound combinations.")
